# Tuning the Edge+Cat Model

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import gc
from collections import Counter
import os
import kagglehub
import torch
import torch.nn as nn
from tqdm import trange
from sklearn.metrics import roc_auc_score, average_precision_score
!pip -q install catboost lightgbm
from catboost import CatBoostClassifier

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 26.8 MB/s eta 0:00:00


In [ ]:
def null_summary(df):
    null_counts = df.isna().sum()

    summary = pd.DataFrame({
        "null_count": null_counts,
    })

    return summary[summary["null_count"] > 0]

def duplicate_check(df):
    total = len(df)
    dup_all = df.duplicated().sum()

    return {
        "total_rows": total,
        "duplicate_rows": dup_all,
        "duplicate_rate": dup_all / total
    }

def timestamp_summary(df, time_col="Timestamp"):
    ts = pd.to_datetime(df[time_col], errors="coerce")

    return {
        "min_time": ts.min(),
        "max_time": ts.max(),
        "null_timestamps": ts.isna().sum()
    }

## Using the LI-Small dataset

In [ ]:
kagglehub.dataset_download("ealtman2019/ibm-transactions-for-anti-money-laundering-aml", path="LI-Small_Trans.csv")

100%|██████████| 620M/620M [00:40<00:00, 16.1MB/s]


'/root/.cache/kagglehub/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/versions/8/LI-Small_Trans.csv'

In [ ]:
BASE = "/root/.cache/kagglehub/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/versions/8"
LI_Strans = pd.read_csv(f"{BASE}/LI-Small_Trans.csv")
LI_Strans.head(5)

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:08,11,8000ECA90,11,8000ECA90,3195403.00,US Dollar,3195403.00,US Dollar,Reinvestment,0
1,2022/09/01 00:21,3402,80021DAD0,3402,80021DAD0,1858.96,US Dollar,1858.96,US Dollar,Reinvestment,0
2,2022/09/01 00:00,11,8000ECA90,1120,8006AA910,592571.00,US Dollar,592571.00,US Dollar,Cheque,0
3,2022/09/01 00:16,3814,8006AD080,3814,8006AD080,12.32,US Dollar,12.32,US Dollar,Reinvestment,0
4,2022/09/01 00:00,20,8006AD530,20,8006AD530,2941.56,US Dollar,2941.56,US Dollar,Reinvestment,0


In [ ]:
LI_Strans["Is Laundering"].unique() #checking the cleanliness of the "Is Laundering" column; all good

array([0, 1])

In [ ]:
null_summary(LI_Strans) #checking for any null values; all good

,null_count


In [ ]:
duplicate_check(LI_Strans) #checking for any duplicates; 8 duplicates

{'total_rows': 6924049,
 'duplicate_rows': np.int64(8),
 'duplicate_rate': np.float64(1.1553933254949525e-06)}

In [ ]:
#dropping the duplicates
print(f"Shape before removing duplicates: {LI_Strans.shape}")
LI_Strans = LI_Strans.drop_duplicates().reset_index(drop=True)
print(f"New shape after removing duplicates: {LI_Strans.shape}")

Shape before removing duplicates: (6924049, 11)
New shape after removing duplicates: (6924041, 11)


In [ ]:
# Check for invalid account names
account_cols = [c for c in LI_Strans.columns if "account" in c.lower()]

if len(account_cols) == 0:
    print("No account columns found (no columns containing 'account').")
else:
    # Basic validity rules:
    # - not null
    # - not empty/whitespace
    # - only allows letters, digits, underscore, hyphen, dot (customize if needed)
    allowed_pattern = r"^[A-Za-z0-9_.-]+$"

    invalid_summary = {}

    for c in account_cols:
        s = LI_Strans[c].astype("string")

        is_null = s.isna()
        is_empty = s.str.strip().eq("")
        bad_chars = ~s.str.match(allowed_pattern, na=False)

        invalid_mask = is_null | is_empty | bad_chars
        invalid_count = int(invalid_mask.sum())

        invalid_summary[c] = {
            "invalid_count": invalid_count,
            "null_count": int(is_null.sum()),
            "empty_count": int(is_empty.sum()),
            "bad_char_count": int(bad_chars.sum())
        }

        print(f"\n[{c}] invalid rows: {invalid_count}")
        print("  null:", invalid_summary[c]["null_count"])
        print("  empty:", invalid_summary[c]["empty_count"])
        print("  bad_chars:", invalid_summary[c]["bad_char_count"])

        if invalid_count > 0:
            examples = LI_Strans.loc[invalid_mask, c].astype("string").head(10).tolist()
            print("  examples:", examples)

    print("\nChecked account columns:", account_cols)


[Account] invalid rows: 0
  null: 0
  empty: 0
  bad_chars: 0

[Account.1] invalid rows: 0
  null: 0
  empty: 0
  bad_chars: 0

Checked account columns: ['Account', 'Account.1']


In [ ]:
# Check for invalid transaction values

value_cols = [c for c in LI_Strans.columns if any(k in c.lower() for k in ["amount", "value"])]

if len(value_cols) == 0:
    print("No transaction value columns found (no columns containing 'amount' or 'value').")
else:
    for c in value_cols:
        x = pd.to_numeric(LI_Strans[c], errors="coerce")

        is_nan = x.isna()
        is_inf = np.isinf(x.to_numpy(dtype=float, copy=False))
        is_neg = x < 0
        is_zero = x == 0

        invalid_mask = is_nan | is_inf | is_neg

        print(f"\n[{c}]")
        print("  total rows:", len(LI_Strans))
        print("  NaN after numeric coercion:", int(is_nan.sum()))
        print("  inf:", int(is_inf.sum()))
        print("  negative:", int(is_neg.sum()))
        print("  zero:", int(is_zero.sum()))
        print("  invalid (NaN/inf/negative):", int(invalid_mask.sum()))

        if int(invalid_mask.sum()) > 0:
            example_rows = LI_Strans.loc[invalid_mask, [c]].head(10)
            print("  first invalid examples:")
            display(example_rows)

    print("\nChecked value columns:", value_cols)


[Amount Received]
  total rows: 6924041
  NaN after numeric coercion: 0
  inf: 0
  negative: 0
  zero: 0
  invalid (NaN/inf/negative): 0

[Amount Paid]
  total rows: 6924041
  NaN after numeric coercion: 0
  inf: 0
  negative: 0
  zero: 0
  invalid (NaN/inf/negative): 0

Checked value columns: ['Amount Received', 'Amount Paid']


In [ ]:
amt = pd.to_numeric(LI_Strans["Amount Received"], errors="coerce")
LI_Strans["Log Amount Received"] = np.log1p(amt)

In [ ]:
timestamp_summary(LI_Strans)

{'min_time': Timestamp('2022-09-01 00:00:00'),
 'max_time': Timestamp('2022-09-17 15:28:00'),
 'null_timestamps': np.int64(0)}

In [ ]:
# Setup for date data preprocessing

TS_COL = "Timestamp"
LABEL_COL = "Is Laundering"

TS_FORMAT = "%Y/%m/%d %H:%M"

# Use True for Large
USE_STREAMING = False
CHUNKSIZE = 2_000_000

CSV_PATH = "/root/.cache/kagglehub/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/versions/8/LI-Small_Trans.csv"

def _coerce_label_to_int(series):
    return pd.to_numeric(series.astype(str).str.strip(), errors="coerce").fillna(0).astype(int)


def _time_aggs_from_df(df_in, ts_col=TS_COL, label_col=LABEL_COL, ts_format=TS_FORMAT):
    y = _coerce_label_to_int(df_in[label_col]).to_numpy(dtype=np.int8)

    if ts_format is None:
        ts = pd.to_datetime(df_in[ts_col], errors="coerce")
    else:
        ts = pd.to_datetime(df_in[ts_col], format=ts_format, errors="coerce")

    mask = ts.notna().to_numpy()
    bad_ts = int((~mask).sum())

    ts = ts[mask]
    y = y[mask]

    hour = ts.dt.hour.to_numpy(dtype=np.int16)
    dow = ts.dt.dayofweek.to_numpy(dtype=np.int16)  # Mon=0..Sun=6

    hour_total = np.bincount(hour, minlength=24).astype(np.int64)
    hour_pos   = np.bincount(hour, weights=y, minlength=24).astype(np.int64)

    dow_total = np.bincount(dow, minlength=7).astype(np.int64)
    dow_pos   = np.bincount(dow, weights=y, minlength=7).astype(np.int64)

    idx = dow * 24 + hour
    hd_total = np.bincount(idx, minlength=7*24).reshape(7, 24).astype(np.int64)
    hd_pos   = np.bincount(idx, weights=y, minlength=7*24).reshape(7, 24).astype(np.int64)

    return {
        "hour_total": hour_total, "hour_pos": hour_pos,
        "dow_total": dow_total,   "dow_pos": dow_pos,
        "hd_total": hd_total,     "hd_pos": hd_pos,
        "total_rows": int(mask.sum()),
        "bad_ts": bad_ts
    }


def _time_aggs_streaming(csv_path, chunksize=CHUNKSIZE, ts_col=TS_COL, label_col=LABEL_COL, ts_format=TS_FORMAT):
    hour_total = np.zeros(24, dtype=np.int64)
    hour_pos   = np.zeros(24, dtype=np.int64)

    dow_total  = np.zeros(7, dtype=np.int64)
    dow_pos    = np.zeros(7, dtype=np.int64)

    hd_total   = np.zeros((7, 24), dtype=np.int64)
    hd_pos     = np.zeros((7, 24), dtype=np.int64)

    total_rows = 0
    bad_ts = 0

    usecols = [ts_col, label_col]

    for chunk in pd.read_csv(csv_path, usecols=usecols, chunksize=chunksize):
        total_rows += len(chunk)

        y = _coerce_label_to_int(chunk[label_col]).to_numpy(dtype=np.int8)

        if ts_format is None:
            ts = pd.to_datetime(chunk[ts_col], errors="coerce")
        else:
            ts = pd.to_datetime(chunk[ts_col], format=ts_format, errors="coerce")

        mask = ts.notna().to_numpy()
        if not mask.all():
            bad_ts += int((~mask).sum())

        ts = ts[mask]
        y  = y[mask]

        hour = ts.dt.hour.to_numpy(dtype=np.int16)
        dow  = ts.dt.dayofweek.to_numpy(dtype=np.int16)

        hour_total += np.bincount(hour, minlength=24)
        hour_pos   += np.bincount(hour, weights=y, minlength=24).astype(np.int64)

        dow_total += np.bincount(dow, minlength=7)
        dow_pos   += np.bincount(dow, weights=y, minlength=7).astype(np.int64)

        idx = dow * 24 + hour
        flat_total = np.bincount(idx, minlength=7*24).reshape(7, 24)
        flat_pos   = np.bincount(idx, weights=y, minlength=7*24).reshape(7, 24).astype(np.int64)

        hd_total += flat_total
        hd_pos   += flat_pos

    return {
        "hour_total": hour_total, "hour_pos": hour_pos,
        "dow_total": dow_total,   "dow_pos": dow_pos,
        "hd_total": hd_total,     "hd_pos": hd_pos,
        "total_rows": int(total_rows),
        "bad_ts": int(bad_ts)
    }

In [ ]:
if USE_STREAMING:
    agg = _time_aggs_streaming(CSV_PATH, chunksize=CHUNKSIZE, ts_format=TS_FORMAT)
else:
    agg = _time_aggs_from_df(LI_Strans, ts_format=TS_FORMAT)

hour_total = agg["hour_total"]
hour_pos   = agg["hour_pos"]
dow_total  = agg["dow_total"]
dow_pos    = agg["dow_pos"]
hd_total   = agg["hd_total"]
hd_pos     = agg["hd_pos"]

print("Temporal EDA for:", "LI-Small_Trans.csv")
print("Rows used (valid timestamps):", agg["total_rows"])
print("Bad/unparsed timestamps:", agg["bad_ts"])
print("Overall laundering rate:", float(hour_pos.sum() / max(hour_total.sum(), 1)))

Temporal EDA for: LI-Small_Trans.csv
Rows used (valid timestamps): 6924041
Bad/unparsed timestamps: 0
Overall laundering rate: 0.0005148727455542219


In [ ]:
hour_df = pd.DataFrame({
    "hour": np.arange(24),
    "tx_count": hour_total,
    "laundering_count": hour_pos,
    "laundering_rate": hour_pos / np.maximum(hour_total, 1)
})

dow_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
dow_df = pd.DataFrame({
    "dow": np.arange(7),
    "day": dow_names,
    "tx_count": dow_total,
    "laundering_count": dow_pos,
    "laundering_rate": dow_pos / np.maximum(dow_total, 1)
})

hd_rate = hd_pos / np.maximum(hd_total, 1)

In [ ]:
if TS_FORMAT is None:
    ts = pd.to_datetime(LI_Strans[TS_COL], errors="coerce")
else:
    ts = pd.to_datetime(LI_Strans[TS_COL], format=TS_FORMAT, errors="coerce")

LI_Strans["_ts"] = ts
LI_Strans["tx_hour"] = LI_Strans["_ts"].dt.hour.astype("Int16")
LI_Strans["tx_dow"] = LI_Strans["_ts"].dt.dayofweek.astype("Int16")     # Mon=0..Sun=6
LI_Strans["tx_month"] = LI_Strans["_ts"].dt.month.astype("Int16")
LI_Strans["tx_day"] = LI_Strans["_ts"].dt.day.astype("Int16")
LI_Strans["tx_date"] = LI_Strans["_ts"].dt.date                       # python date
LI_Strans["tx_is_weekend"] = LI_Strans["tx_dow"].isin([5, 6]).astype("Int8")
LI_Strans["tx_hour_sin"] = np.sin(2 * np.pi * LI_Strans["tx_hour"].fillna(0) / 24.0)
LI_Strans["tx_hour_cos"] = np.cos(2 * np.pi * LI_Strans["tx_hour"].fillna(0) / 24.0)

In [ ]:
LI_Strans.head()

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,...,Log Amount Received,_ts,tx_hour,tx_dow,tx_month,tx_day,tx_date,tx_is_weekend,tx_hour_sin,tx_hour_cos
0,2022/09/01 00:08,11,8000ECA90,11,8000ECA90,3195403.00,US Dollar,3195403.00,US Dollar,Reinvestment,...,14.977224,2022-09-01 00:08:00,0,3,9,1,2022-09-01,0,0.0,1.0
1,2022/09/01 00:21,3402,80021DAD0,3402,80021DAD0,1858.96,US Dollar,1858.96,US Dollar,Reinvestment,...,7.528310,2022-09-01 00:21:00,0,3,9,1,2022-09-01,0,0.0,1.0
2,2022/09/01 00:00,11,8000ECA90,1120,8006AA910,592571.00,US Dollar,592571.00,US Dollar,Cheque,...,13.292228,2022-09-01 00:00:00,0,3,9,1,2022-09-01,0,0.0,1.0
3,2022/09/01 00:16,3814,8006AD080,3814,8006AD080,12.32,US Dollar,12.32,US Dollar,Reinvestment,...,2.589267,2022-09-01 00:16:00,0,3,9,1,2022-09-01,0,0.0,1.0
4,2022/09/01 00:00,20,8006AD530,20,8006AD530,2941.56,US Dollar,2941.56,US Dollar,Reinvestment,...,7.987035,2022-09-01 00:00:00,0,3,9,1,2022-09-01,0,0.0,1.0


### Modeling

In [ ]:
df0 = LI_Strans

# Must exist from your preprocessing
assert "_ts" in df0.columns, "Expected LI_Strans['_ts'] from preprocessing."
assert "Is Laundering" in df0.columns, "Missing Is Laundering column."

# Keep only modeling columns (edit if you want more features)
keep_cols = [
    "_ts", "Is Laundering",
    "From Bank", "Account", "To Bank", "Account.1",
    "Amount Paid", "Amount Received",
    "Receiving Currency", "Payment Currency", "Payment Format",
    "Log Amount Received",
    "tx_hour", "tx_dow", "tx_is_weekend", "tx_hour_sin", "tx_hour_cos",
]
keep_cols = [c for c in keep_cols if c in df0.columns]
df = df0[keep_cols]

# Label clean (defensive)
y_ser = pd.to_numeric(df["Is Laundering"].astype(str).str.strip(), errors="coerce")
mask = y_ser.isin([0, 1])
df = df.loc[mask].copy()  # copy AFTER narrowing + filtering
df["Is Laundering"] = y_ser.loc[mask].astype(np.float32)

# Drop invalid timestamps
df = df.dropna(subset=["_ts"])

print("rows:", len(df), "pos_rate:", float(df["Is Laundering"].mean()))
print("columns:", df.columns.tolist())

rows: 6924041 pos_rate: 0.0005148727213963866
columns: ['_ts', 'Is Laundering', 'From Bank', 'Account', 'To Bank', 'Account.1', 'Amount Paid', 'Amount Received', 'Receiving Currency', 'Payment Currency', 'Payment Format', 'Log Amount Received', 'tx_hour', 'tx_dow', 'tx_is_weekend', 'tx_hour_sin', 'tx_hour_cos']


In [ ]:
order = np.argsort(df["_ts"].to_numpy())
df = df.iloc[order].reset_index(drop=True)
df["row_id"] = np.arange(len(df), dtype=np.int64)
print("time:", df["_ts"].min(), "->", df["_ts"].max())

time: 2022-09-01 00:00:00 -> 2022-09-17 15:28:00


In [ ]:
N = len(df)
n_train = int(0.70 * N)
n_val   = int(0.15 * N)

tr_idx_np   = np.arange(0, n_train, dtype=np.int64)
val_idx_np  = np.arange(n_train, n_train+n_val, dtype=np.int64)
test_idx_np = np.arange(n_train+n_val, N, dtype=np.int64)

print("Split sizes:", len(tr_idx_np), len(val_idx_np), len(test_idx_np))
print("Pos rates:",
      float(df["Is Laundering"].to_numpy()[tr_idx_np].mean()),
      float(df["Is Laundering"].to_numpy()[val_idx_np].mean()),
      float(df["Is Laundering"].to_numpy()[test_idx_np].mean()))

Split sizes: 4846828 1038606 1038607
Pos rates: 0.00046030106022953987 0.0005613293033093214 0.000723083910997957


In [ ]:
MAX_EDGES = 3_000_000  # adjust upward later if stable

if len(df) > MAX_EDGES:
    df_edge = df.iloc[:MAX_EDGES].copy()
else:
    df_edge = df.copy()

print("EdgeMLP edges used:", len(df_edge), "of", len(df))

EdgeMLP edges used: 3000000 of 6924041


In [ ]:
src_df = df_edge[["From Bank","Account"]]
dst_df = df_edge[["To Bank","Account.1"]].rename(columns={"To Bank":"From Bank","Account.1":"Account"})
all_df = pd.concat([src_df, dst_df], ignore_index=True)

codes, uniques = pd.factorize(list(map(tuple, all_df.to_numpy())))
m = len(df_edge)

src = codes[:m].astype(np.int64)
dst = codes[m:].astype(np.int64)
y   = df_edge["Is Laundering"].to_numpy(dtype=np.float32)

num_nodes = len(uniques)
print("nodes:", num_nodes, "edges:", m, "pos_rate:", float(y.mean()))

/tmp/ipython-input-2499473087.py:5: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  codes, uniques = pd.factorize(list(map(tuple, all_df.to_numpy())))


nodes: 700711 edges: 3000000 pos_rate: 0.00036466665915213525


In [ ]:
class EdgeMLP(nn.Module):
    def __init__(self, n_nodes, d=128, dropout=0.2):
        super().__init__()
        self.emb = nn.Embedding(n_nodes, d)
        self.drop = nn.Dropout(dropout)
        self.net = nn.Sequential(
            nn.Linear(4*d, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
        )

    def forward(self, s, t):
        Hs = self.drop(self.emb(s))
        Hd = self.drop(self.emb(t))
        x = torch.cat([Hs, Hd, (Hs - Hd).abs(), Hs * Hd], dim=1)
        return self.net(x).squeeze(-1)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

src_gpu = torch.from_numpy(src).long().to(device)
dst_gpu = torch.from_numpy(dst).long().to(device)
y_gpu   = torch.from_numpy(y).float().to(device)

BATCH  = 262_144
EPOCHS = 5  # start small on CPU; raise later if stable
N_FOLDS = 5

def fit_edgemlp(train_idx_np, epochs=EPOCHS):
    model = EdgeMLP(num_nodes, d=128, dropout=0.2).to(device)

    pos_rate = float(y[train_idx_np].mean())
    pos_w = (1.0 - pos_rate) / max(pos_rate, 1e-12)
    pos_weight = torch.tensor([min(pos_w, 1000.0)], device=device)  # cap helps stability
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

    for ep in range(epochs):
        model.train()
        order = train_idx_np.copy()
        np.random.shuffle(order)

        running = 0.0
        for i in range(0, len(order), BATCH):
            b_np = order[i:i+BATCH]
            b = torch.from_numpy(b_np).to(device)

            opt.zero_grad(set_to_none=True)
            logits = model(src_gpu[b], dst_gpu[b])
            loss = loss_fn(logits, y_gpu[b])
            loss.backward()
            opt.step()

            running += float(loss.detach()) * len(b_np)

        print(f"EdgeMLP Epoch {ep:02d} | loss={running/len(order):.6f}")

    return model

@torch.no_grad()
def predict_edgemlp(model, idx_np):
    model.eval()
    out = np.empty(len(idx_np), dtype=np.float32)
    for i in range(0, len(idx_np), BATCH):
        b_np = idx_np[i:i+BATCH]
        b = torch.from_numpy(b_np).to(device)
        out[i:i+len(b_np)] = torch.sigmoid(model(src_gpu[b], dst_gpu[b])).cpu().numpy().astype(np.float32)
    return out
def eval_edgemlp(model, idx_np, name="split"):
    p = predict_edgemlp(model, idx_np)
    yt = y[idx_np]
    roc = roc_auc_score(yt, p) if yt.sum() > 0 else float("nan")
    pr  = average_precision_score(yt, p) if yt.sum() > 0 else float("nan")
    print(f"[EdgeMLP-only] {name}: ROC={roc:.6f}  PR={pr:.6f}  base_rate={yt.mean():.6f}  pred_mean={p.mean():.6f}")
    return roc, pr, p

device: cuda


In [ ]:
N_edge = len(df_edge)

# Time split indices for df_edge
n_train_edge = min(int(0.70 * len(df)), N_edge)  # align with original split boundary
n_val_edge   = min(int(0.15 * len(df)), max(0, N_edge - n_train_edge))

tr_edge = np.arange(0, n_train_edge, dtype=np.int64)
va_edge = np.arange(n_train_edge, n_train_edge + n_val_edge, dtype=np.int64)
te_edge = np.arange(n_train_edge + n_val_edge, N_edge, dtype=np.int64)

print("EdgeMLP split sizes:", len(tr_edge), len(va_edge), len(te_edge))
print("EdgeMLP pos rates:", float(y[tr_edge].mean()), float(y[va_edge].mean()) if len(va_edge)>0 else None)

edge_score_edge = np.full(N_edge, np.nan, dtype=np.float32)

# Contiguous time blocks within TRAIN for OOF
blocks = np.array_split(tr_edge, N_FOLDS)

print("OOF EdgeMLP scores on TRAIN (time-respecting blocks)...")
for k in range(N_FOLDS):
    holdout = blocks[k]

    # SAFE FIX: skip first block (no earlier data)
    if k == 0:
        print(f"Block {k+1}/{N_FOLDS}: skipped (no earlier data).")
        continue

    train_f = np.concatenate(blocks[:k])  # now guaranteed non-empty

    m_k = fit_edgemlp(train_f, epochs=EPOCHS)
    edge_score_edge[holdout] = predict_edgemlp(m_k, holdout)

    print(f"Block {k+1}/{N_FOLDS} done.")

print("Final EdgeMLP on full TRAIN → score VAL/TEST + fill skipped TRAIN block if any...")
m_final = fit_edgemlp(tr_edge, epochs=EPOCHS)

if len(va_edge) > 0:
    edge_val_roc, edge_val_pr, edge_val_probs = eval_edgemlp(m_final, va_edge, name="VAL")
else:
    edge_val_roc, edge_val_pr, edge_val_probs = float("nan"), float("nan"), None

if len(te_edge) > 0:
    edge_test_roc, edge_test_pr, edge_test_probs = eval_edgemlp(m_final, te_edge, name="TEST")
else:
    edge_test_roc, edge_test_pr, edge_test_probs = float("nan"), float("nan"), None
# Fill any skipped earliest block (optional; see note below)
nan_tr = tr_edge[np.isnan(edge_score_edge[tr_edge])]
if len(nan_tr) > 0:
    edge_score_edge[nan_tr] = predict_edgemlp(m_final, nan_tr)

if len(va_edge) > 0:
    edge_score_edge[va_edge] = predict_edgemlp(m_final, va_edge)
if len(te_edge) > 0:
    edge_score_edge[te_edge] = predict_edgemlp(m_final, te_edge)

print("NaNs remaining (TRAIN/VAL/TEST):",
      np.isnan(edge_score_edge[tr_edge]).sum(),
      np.isnan(edge_score_edge[va_edge]).sum() if len(va_edge)>0 else 0,
      np.isnan(edge_score_edge[te_edge]).sum() if len(te_edge)>0 else 0)

# Attach edge scores back to df_edge and then map into df (full) by row order
df_edge["edge_score"] = edge_score_edge

EdgeMLP split sizes: 3000000 0 0
EdgeMLP pos rates: 0.00036466665915213525 None
OOF EdgeMLP scores on TRAIN (time-respecting blocks)...
Block 1/5: skipped (no earlier data).
EdgeMLP Epoch 00 | loss=0.739536
EdgeMLP Epoch 01 | loss=0.377319
EdgeMLP Epoch 02 | loss=0.301848
EdgeMLP Epoch 03 | loss=0.272456
EdgeMLP Epoch 04 | loss=0.220298
Block 2/5 done.
EdgeMLP Epoch 00 | loss=0.650686
EdgeMLP Epoch 01 | loss=0.455854
EdgeMLP Epoch 02 | loss=0.394467
EdgeMLP Epoch 03 | loss=0.385029
EdgeMLP Epoch 04 | loss=0.372305
Block 3/5 done.
EdgeMLP Epoch 00 | loss=0.684525
EdgeMLP Epoch 01 | loss=0.501289
EdgeMLP Epoch 02 | loss=0.476189
EdgeMLP Epoch 03 | loss=0.456650
EdgeMLP Epoch 04 | loss=0.438582
Block 4/5 done.
EdgeMLP Epoch 00 | loss=0.646214
EdgeMLP Epoch 01 | loss=0.538764
EdgeMLP Epoch 02 | loss=0.520814
EdgeMLP Epoch 03 | loss=0.509977
EdgeMLP Epoch 04 | loss=0.492399
Block 5/5 done.
Final EdgeMLP on full TRAIN → score VAL/TEST + fill skipped TRAIN block if any...
EdgeMLP Epoch 00 | l

In [ ]:
df["edge_score"] = np.nan

# Because df_edge is the first MAX_EDGES rows of df (time-sorted), row alignment is direct:
df.loc[:len(df_edge)-1, "edge_score"] = df_edge["edge_score"].to_numpy()

print("edge_score filled rows:", df["edge_score"].notna().sum(), "out of", len(df))

edge_score filled rows: 3000000 out of 6924041


In [ ]:
df["Amount Paid"] = pd.to_numeric(df["Amount Paid"], errors="coerce").fillna(0.0)
df["Amount Received"] = pd.to_numeric(df["Amount Received"], errors="coerce").fillna(0.0)

# Use your Log Amount Received if present
if "Log Amount Received" in df.columns:
    df["log_amt_recv"] = pd.to_numeric(df["Log Amount Received"], errors="coerce")
else:
    df["log_amt_recv"] = np.log1p(df["Amount Received"].astype(np.float32))

###---I TRIED TUNING THE EDGE+CAT MODEL BY ADDING THESE FEATURES TO BE CONSIDERED WHILE TRAINING THE CATBOOST PORTION (NO IMPROVEMENT)---###

#df["log_amt_paid"] = np.log1p(df["Amount Paid"].astype(np.float32))
#df["cross_bank"] = (df["From Bank"].astype(str) != df["To Bank"].astype(str)).astype(np.int8)

#df["wknd_edge"] = df["tx_is_weekend"].astype(np.float32) * df["edge_score"].astype(np.float32)

#df["wknd_log_recv"] = df["tx_is_weekend"].astype(np.float32) * df["log_amt_recv"].astype(np.float32)
#df["wknd_log_paid"] = df["tx_is_weekend"].astype(np.float32) * df["log_amt_paid"].astype(np.float32)

#df["wknd_cross"] = df["tx_is_weekend"].astype(np.float32) * df["cross_bank"].astype(np.float32)

#df["is_off_hours"] = ((df["tx_hour"] < 6) | (df["tx_hour"] >= 20)).astype(np.int8)

#df["offhrs_edge"] = df["is_off_hours"].astype(np.float32) * df["edge_score"].astype(np.float32)
#df["wknd_offhrs_edge"] = (df["tx_is_weekend"].astype(np.float32)
#                          * df["is_off_hours"].astype(np.float32)
#                          * df["edge_score"].astype(np.float32))

cat_cols = ["From Bank", "To Bank", "Receiving Currency", "Payment Currency", "Payment Format"]
num_cols = [
    "edge_score",
    "log_amt_paid", "log_amt_recv", "cross_bank",
    "tx_hour", "tx_dow", "tx_is_weekend", "tx_hour_sin", "tx_hour_cos"
]

###---HAD TO MODIFY MY NUMERIC FEATURES ADDED IN THE CATBOOST TRAINING AS WELL---###

#num_cols = [
#    "edge_score",
#    "log_amt_paid", "log_amt_recv", "cross_bank",
#    "tx_hour", "tx_dow", "tx_is_weekend", "tx_hour_sin", "tx_hour_cos",

    # interactions
#    "wknd_edge", "wknd_log_recv", "wknd_log_paid", "wknd_cross",
#    "is_off_hours", "offhrs_edge", "wknd_offhrs_edge",
#]

feature_cols = cat_cols + num_cols



# Drop rows where edge_score is missing (rows beyond MAX_EDGES or skipped fold)
df_cb = df.dropna(subset=["edge_score"]).copy()

X = df_cb[feature_cols].copy()
y_cb = df_cb["Is Laundering"].astype(int).to_numpy(np.int32)

# Recompute splits in df_cb by time order position (still time-respecting, but on the filtered df_cb)
n = len(df_cb)
n_train = int(0.70 * n)
n_val   = int(0.15 * n)

Xtr, ytr = X.iloc[:n_train], y_cb[:n_train]
Xva, yva = X.iloc[n_train:n_train+n_val], y_cb[n_train:n_train+n_val]
Xte, yte = X.iloc[n_train+n_val:], y_cb[n_train+n_val:]

print("CatBoost sizes:", Xtr.shape, Xva.shape, Xte.shape)
print("CatBoost pos rates:", ytr.mean(), yva.mean(), yte.mean())

CatBoost sizes: (2100000, 14) (450000, 14) (450000, 14)
CatBoost pos rates: 0.00024190476190476192 0.00033777777777777777 0.0009644444444444444


In [ ]:
cat_idx = [Xtr.columns.get_loc(c) for c in cat_cols]

cb = CatBoostClassifier(
    iterations=3000,
    learning_rate=0.05,
    depth=8,
    loss_function="Logloss",
    eval_metric="PRAUC",
    random_seed=0,
    verbose=200,
    auto_class_weights="Balanced",
    task_type="CPU"
)

cb.fit(Xtr, ytr, eval_set=(Xva, yva), cat_features=cat_idx, use_best_model=True)

pva = cb.predict_proba(Xva)[:, 1]
pte = cb.predict_proba(Xte)[:, 1]

print("\n=== EdgeMLP → CatBoost (time split, leak-free stacking) ===")
print("VAL  ROC:", roc_auc_score(yva, pva))
print("VAL  PR :", average_precision_score(yva, pva))
print("TEST ROC:", roc_auc_score(yte, pte))
print("TEST PR :", average_precision_score(yte, pte))

###---DECIDED TO CONDUCT HYPERPARAMETER TUNING VIA GRID SEARCH; NOT EVEN THE BEST HYPERPARAMETERS COMPARE TO OUR RESULTS FROM OUR GNN MODELS---###

def fit_eval_catboost(params, Xtr, ytr, Xva, yva, cat_idx, seed=0):
    model = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="PRAUC",          # focus on PR-AUC
        random_seed=seed,
        task_type="CPU",
        verbose=False,
        early_stopping_rounds=50,
        **params
    )
    model.fit(Xtr, ytr, eval_set=(Xva, yva), cat_features=cat_idx, use_best_model=True)

    pva = model.predict_proba(Xva)[:, 1]
    out = {
        **params,
        "best_iter": int(model.get_best_iteration()),
        "val_pr": float(average_precision_score(yva, pva)),
        "val_roc": float(roc_auc_score(yva, pva)),
        "val_base": float(np.mean(yva)),
        "val_lift": float(average_precision_score(yva, pva) / max(np.mean(yva), 1e-12)),
    }
    return model, out

from itertools import product

grid = {
    "depth": [6, 8, 10],
    "learning_rate": [0.03, 0.05, 0.1],
    "l2_leaf_reg": [3, 10],
    "iterations": [5000],                 # rely on early stopping
    "auto_class_weights": ["Balanced"],   # keep consistent
}

results = []
best = None
best_model = None

total = len(grid["depth"]) * len(grid["learning_rate"]) * len(grid["l2_leaf_reg"])
run_id = 0

for depth, lr, l2 in product(grid["depth"], grid["learning_rate"], grid["l2_leaf_reg"]):
    run_id += 1
    params = {
        "depth": depth,
        "learning_rate": lr,
        "l2_leaf_reg": l2,
        "iterations": grid["iterations"][0],
        "auto_class_weights": grid["auto_class_weights"][0],
    }

    model, row = fit_eval_catboost(params, Xtr, ytr, Xva, yva, cat_idx, seed=0)
    results.append(row)

    print(f"[{run_id:02d}/{total}] depth={depth} lr={lr} l2={l2} "
          f"| val_PR={row['val_pr']:.6f} (lift={row['val_lift']:.1f}x) "
          f"| val_ROC={row['val_roc']:.6f} | best_iter={row['best_iter']}")

    if (best is None) or (row["val_pr"] > best["val_pr"]):
        best = row
        best_model = model

res_df = pd.DataFrame(results).sort_values(["val_pr", "val_roc"], ascending=False).reset_index(drop=True)
print("\nTop 10 configs:")
display(res_df.head(10))

print("\nBest config:", best)

# Evaluate best model on TEST
p_test = best_model.predict_proba(Xte)[:, 1]
test_pr = average_precision_score(yte, p_test)
test_roc = roc_auc_score(yte, p_test)
test_base = float(np.mean(yte))
test_lift = float(test_pr / max(test_base, 1e-12))

print("\n=== BEST MODEL TEST ===")
print(f"TEST PR-AUC: {test_pr:.6f} (lift={test_lift:.1f}x over base {test_base:.6f})")
print(f"TEST ROC-AUC: {test_roc:.6f}")

# Save grid results
res_df.to_csv("catboost_grid_results.csv", index=False)
print("\nSaved → catboost_grid_results.csv")

[01/18] depth=6 lr=0.03 l2=3 | val_PR=0.013095 (lift=38.8x) | val_ROC=0.940986 | best_iter=297
[02/18] depth=6 lr=0.03 l2=10 | val_PR=0.013495 (lift=40.0x) | val_ROC=0.939996 | best_iter=295
[03/18] depth=6 lr=0.05 l2=3 | val_PR=0.007736 (lift=22.9x) | val_ROC=0.938351 | best_iter=156
[04/18] depth=6 lr=0.05 l2=10 | val_PR=0.010256 (lift=30.4x) | val_ROC=0.939238 | best_iter=140
[05/18] depth=6 lr=0.1 l2=3 | val_PR=0.008266 (lift=24.5x) | val_ROC=0.936006 | best_iter=65
[06/18] depth=6 lr=0.1 l2=10 | val_PR=0.006603 (lift=19.5x) | val_ROC=0.935864 | best_iter=118
[07/18] depth=8 lr=0.03 l2=3 | val_PR=0.006704 (lift=19.8x) | val_ROC=0.938847 | best_iter=196
[08/18] depth=8 lr=0.03 l2=10 | val_PR=0.008998 (lift=26.6x) | val_ROC=0.935720 | best_iter=241
[09/18] depth=8 lr=0.05 l2=3 | val_PR=0.006865 (lift=20.3x) | val_ROC=0.937093 | best_iter=93
[10/18] depth=8 lr=0.05 l2=10 | val_PR=0.009210 (lift=27.3x) | val_ROC=0.934147 | best_iter=112
[11/18] depth=8 lr=0.1 l2=3 | val_PR=0.013129 (li

,depth,learning_rate,l2_leaf_reg,iterations,auto_class_weights,best_iter,val_pr,val_roc,val_base,val_lift
0,6,0.03,10,5000,Balanced,295,0.013495,0.939996,0.000338,39.952252
1,8,0.10,3,5000,Balanced,86,0.013129,0.935335,0.000338,38.869374
2,6,0.03,3,5000,Balanced,297,0.013095,0.940986,0.000338,38.767802
3,10,0.05,3,5000,Balanced,150,0.010487,0.939296,0.000338,31.048084
4,6,0.05,10,5000,Balanced,140,0.010256,0.939238,0.000338,30.364367
5,8,0.05,10,5000,Balanced,112,0.009210,0.934147,0.000338,27.266141
6,8,0.03,10,5000,Balanced,241,0.008998,0.935720,0.000338,26.639089
7,10,0.10,10,5000,Balanced,133,0.008994,0.935204,0.000338,26.626578
8,6,0.10,3,5000,Balanced,65,0.008266,0.936006,0.000338,24.470946
9,6,0.05,3,5000,Balanced,156,0.007736,0.938351,0.000338,22.903946



Best config: {'depth': 6, 'learning_rate': 0.03, 'l2_leaf_reg': 10, 'iterations': 5000, 'auto_class_weights': 'Balanced', 'best_iter': 295, 'val_pr': 0.013494983044772634, 'val_roc': 0.9399962013484788, 'val_base': 0.00033777777777777777, 'val_lift': 39.95225243518214}

=== BEST MODEL TEST ===
TEST PR-AUC: 0.029257 (lift=30.3x over base 0.000964)
TEST ROC-AUC: 0.943268

Saved → catboost_grid_results.csv


###The results were worse for PR:



DATASET_NAME = "LI-Small"          # <-- change each run
MODEL_NAME   = "EdgeMLP+CatBoost"
SPLIT_NAME   = "time_70_15_15"

val_roc = roc_auc_score(yva, pva)
test_roc = roc_auc_score(yte, pte)

val_pr = average_precision_score(yva, pva)
test_pr = average_precision_score(yte, pte)

val_base = float(yva.mean())
test_base = float(yte.mean())

row = {
    "dataset": DATASET_NAME,
    "model": MODEL_NAME,
    "split": SPLIT_NAME,
    "val_roc": val_roc,
    "test_roc": test_roc,
    "val_pr_auc": val_pr,
    "test_pr_auc": test_pr,
    "val_base_rate": val_base,
    "test_base_rate": test_base,
    "val_pr_lift": val_pr / val_base if val_base > 0 else None,
    "test_pr_lift": test_pr / test_base if test_base > 0 else None,
    "n_val": int(len(yva)),
    "n_test": int(len(yte)),
}

new_df = pd.DataFrame([row])

path = Path("aml_model_results.csv")
if path.exists():
    old = pd.read_csv(path)
    out = pd.concat([old, new_df], ignore_index=True)
else:
    out = new_df

out.to_csv(path, index=False)
print(f"Saved → {path} (rows={len(out)})")
out.tail(10)